# Beijing order-fraction demonstration
Replication target: the Appendix C observation that real dependence is less
absorbing than the Gaussian-copula reference at equal Pearson correlation.

Data: multi-site Beijing air-quality panel (12 sites x 35,064 hourly rows x
11 standardized columns, no names stored). Column-order hypothesis, confirmed
by signature diagnostics (missingness, ranges, kurtosis, correlation
fingerprints): PM2.5, PM10, SO2, NO2, CO, O3, TEMP, PRES, DEWP,
RAIN, WSPM. Triple: TEMP (col 6), PRES (col 7), WSPM (col 10) -- one strong
NEGATIVE pair (TEMP-PRES, pooled Pearson about -0.83) and a third variable
nearly independent of both (+0.03, +0.06). The closed-form law is even in
rho, so the single-pair reference applies unchanged at |rho|.

Contrasts with the HDL demo: benign marginals (kurtosis about 2/2/7 -- no
sparse-count pathology), so the permuted product anchors are RESTORED to
acceptance checks here; and a near-collapse reference point
(F(0.83) about 0.024 vs HDL's 0.178), probing the regime where the
reference approaches estimator resolution.

Design: pooled complete cases across sites; per seed, ONE seeded 60,000-row
subsample shared by all three conditions (real standardized; rank-Gaussianized;
column-permuted Gaussianized). Targets and estimators identical to
vdem_order_demo. Acceptance checks (pre-registered): (1) pairwise control
RBF < 0.02 every condition and seed (population 0 provable; threshold
empirical class-adequacy); (2)+(3) permuted centered monomial and tanh
product, OOS poly in (0.85, 1.15) each seed (population 1 provable under
independence + centering). No real-outcome arm.

Outputs to `MyDrive/KDD_Interactions/results/beijing_order_demo/`.
Data expected at `MyDrive/KDD_Interactions/data/beijing_panel.npz`
(falls back to `MyDrive/GNAVAR/data/`).


In [ ]:
# Cell 1 -- Mount Drive, locate data, set up output folder
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/KDD_Interactions'
OUT = os.path.join(BASE, 'results', 'beijing_order_demo')
os.makedirs(OUT, exist_ok=True)
CANDIDATES = [
    os.path.join(BASE, 'data', 'beijing_panel.npz'),
    '/content/drive/MyDrive/GNAVAR/data/beijing_panel.npz',
]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_PATH, f"beijing_panel.npz not found in {CANDIDATES}; copy it to one of these paths"
print('data:', DATA_PATH)
print('output folder:', OUT)


In [ ]:
# Cell 2 -- Load, pool sites, complete cases, correlations
import numpy as np, csv, json, time, hashlib

COLNAMES = ["PM2.5","PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM"]
TRIPLE_IDX = [6, 7, 10]                       # TEMP, PRES, WSPM
TRIPLE = [COLNAMES[j] for j in TRIPLE_IDX]

z = np.load(DATA_PATH, allow_pickle=True)
site_keys = sorted(k for k in z.keys() if k.startswith("X__"))
assert len(site_keys) == 12, f"expected 12 sites, found {len(site_keys)}"
pooled = np.concatenate([z[k] for k in site_keys], axis=0).astype(float)
assert pooled.shape[1] == 11, f"expected 11 columns, found {pooled.shape[1]}"
X_all = pooled[:, TRIPLE_IDX]
ok = ~np.isnan(X_all).any(axis=1)
X_all = X_all[ok]
print(f"sites: {len(site_keys)}, pooled rows: {len(pooled)}, triple-complete: {len(X_all)}")

Cfull = np.corrcoef(((X_all - X_all.mean(0)) / X_all.std(0)).T)
print("pooled Pearson correlations of the triple:")
for i, nm in enumerate(TRIPLE):
    print("  " + nm.ljust(6) + " ".join(f"{Cfull[i,j]:+.3f}" for j in range(3)))
RHO_PAIR = float(Cfull[0, 1])                 # TEMP-PRES (negative)
single_pair_F = lambda r: (1 - r**2) ** 2 / ((1 + r**2) * (1 + 2 * r**2))
print(f"strong pair rho_hat = {RHO_PAIR:+.3f}; law is even in rho: "
      f"reference F(|rho|) = {single_pair_F(abs(RHO_PAIR)):.4f}")


In [ ]:
# Cell 3 -- Estimators and provenance (identical logic to vdem_order_demo)
from itertools import product as iproduct

def monomial_exps(n_vars, D, max_active):
    out = []
    for combo in iproduct(range(D + 1), repeat=n_vars):
        if sum(combo) <= D and sum(1 for c in combo if c > 0) <= max_active:
            out.append(combo)
    return out

def frac_poly(X, h, D, max_active=2):
    exps = monomial_exps(3, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    Phi = np.column_stack(cols)
    mu = Phi.mean(0); sd = Phi.std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd
    Phi[:, exps.index((0, 0, 0))] = 1.0
    hc = h - h.mean()
    denom = float(hc @ hc)
    if denom <= 0: return float("nan")
    beta, *_ = np.linalg.lstsq(Phi, hc, rcond=None)
    r = hc - Phi @ beta
    return float((r @ r) / denom)

def frac_poly_oos(X, h, D, seed, max_active=2, train_frac=0.5):
    n = X.shape[0]
    idx = np.random.default_rng(seed).permutation(n)
    tr, te = idx[: int(train_frac * n)], idx[int(train_frac * n):]
    exps = monomial_exps(3, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(n)
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    Phi = np.column_stack(cols)
    mu = Phi[tr].mean(0); sd = Phi[tr].std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd
    Phi[:, exps.index((0, 0, 0))] = 1.0
    hm = h[tr].mean()
    beta, *_ = np.linalg.lstsq(Phi[tr], h[tr] - hm, rcond=None)
    resid = (h[te] - hm) - Phi[te] @ beta
    denom = np.sum((h[te] - h[te].mean()) ** 2)
    if denom <= 0: return float("nan")
    return float((resid @ resid) / denom)

CENTERS = np.linspace(-2.5, 2.5, 9)
BW = 0.75

def uni_feats(x):
    return np.column_stack([x] + [np.exp(-0.5 * ((x - c) / BW) ** 2) for c in CENTERS])

def design_rbf(X):
    n = X.shape[0]
    U = [uni_feats(X[:, j]) for j in range(3)]
    cols = [np.ones((n, 1))] + U
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        cols.append((U[a][:, :, None] * U[b][:, None, :]).reshape(n, -1))
    return np.concatenate(cols, axis=1)

def random_split(n, seed, train_frac=0.75):
    idx = np.random.default_rng(seed).permutation(n)
    k = int(train_frac * n)
    tr = np.zeros(n, bool); tr[idx[:k]] = True
    return tr, ~tr

def frac_rbf_ridge_split(X, h, lam=10.0, split_seed=0):
    tr, te = random_split(X.shape[0], split_seed)
    Phi = design_rbf(X)
    mu = Phi[tr].mean(0); sd = Phi[tr].std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd; Phi[:, 0] = 1.0
    hm = h[tr].mean()
    P = np.eye(Phi.shape[1]); P[0, 0] = 0.0
    A = Phi[tr].T @ Phi[tr] + lam * P
    b = Phi[tr].T @ (h[tr] - hm)
    try:
        from scipy.linalg import cho_factor, cho_solve
        beta = cho_solve(cho_factor(A), b)
    except Exception:
        beta = np.linalg.solve(A, b)
    resid = (h[te] - hm) - Phi[te] @ beta
    denom = np.sum((h[te] - h[te].mean()) ** 2)
    if denom <= 0: return float("nan")
    return float((resid @ resid) / denom)

def standardize(X):
    sd = X.std(0)
    sd = np.where(sd == 0, 1.0, sd)
    return (X - X.mean(0)) / sd

def normal_scores(X):
    from scipy.stats import rankdata, norm
    Z = np.empty_like(X, dtype=float)
    n = X.shape[0]
    for j in range(X.shape[1]):
        Z[:, j] = norm.ppf((rankdata(X[:, j], method="average") - 0.5) / n)
    return Z

def make_target(Z, name):
    if name == "monomial_c":
        f = [Z[:, j] - Z[:, j].mean() for j in range(3)]
        return f[0] * f[1] * f[2]
    if name == "tanh_prod_c":
        f = [np.tanh(Z[:, j]) - np.tanh(Z[:, j]).mean() for j in range(3)]
        return f[0] * f[1] * f[2]
    if name == "pairwise_control":
        return (np.tanh(Z[:, 0] + Z[:, 1]) + np.tanh(Z[:, 1] + Z[:, 2])
                + np.tanh(Z[:, 0] + Z[:, 2]))
    raise ValueError(name)

N_SUB = 60_000
SEEDS = [0, 1, 2]
RIDGE_LAM = 10.0
TARGETS_B = ["monomial_c", "tanh_prod_c", "pairwise_control"]
CONDITIONS = ["real_std", "gauss_scores", "permuted_gauss"]

_config = {"COLNAMES": COLNAMES, "TRIPLE_IDX": TRIPLE_IDX, "TRIPLE": TRIPLE,
           "N_SUB": N_SUB, "SEEDS": SEEDS, "RIDGE_LAM": RIDGE_LAM,
           "TARGETS_B": TARGETS_B, "CONDITIONS": CONDITIONS,
           "CENTERS": CENTERS.tolist(), "BW": BW, "train_frac": 0.75}
CODE_SHA = hashlib.sha256(
    b"".join(f.__code__.co_code for f in
             [monomial_exps, frac_poly, frac_poly_oos, uni_feats, design_rbf,
              random_split, frac_rbf_ridge_split, standardize, normal_scores,
              make_target])
    + json.dumps(_config, sort_keys=True).encode()).hexdigest()
print("provenance sha256 (bytecode + config):", CODE_SHA)


In [ ]:
# Cell 4 -- Sweep: per seed, one shared subsample across all conditions
EXP = "beijing_order_demo"
t0 = time.time()
rows = []
for s in SEEDS:
    sub = np.random.default_rng(500 + s).permutation(len(X_all))[:N_SUB]
    Xs = X_all[sub]
    Z_real = standardize(Xs)
    Z_gauss = normal_scores(Xs)
    rng = np.random.default_rng(1000 + s)
    Z_perm = Z_gauss.copy()
    for j in range(3):
        Z_perm[:, j] = Z_perm[rng.permutation(N_SUB), j]
    for cond, Z in [("real_std", Z_real), ("gauss_scores", Z_gauss),
                    ("permuted_gauss", Z_perm)]:
        for target in TARGETS_B:
            h = make_target(Z, target)
            rows.append({"experiment": EXP, "condition": cond,
                         "target": target, "seed": s,
                         "frac_poly_D4": frac_poly(Z, h, 4),
                         "frac_poly_oos": frac_poly_oos(Z, h, 4, seed=s),
                         "frac_rbf_ridge": frac_rbf_ridge_split(Z, h, RIDGE_LAM, s)})
        print(f"seed {s} {cond:15s} done  ({time.time()-t0:6.1f}s)", flush=True)

fields = ["experiment","condition","target","seed",
          "frac_poly_D4","frac_poly_oos","frac_rbf_ridge"]
with open(os.path.join(OUT, "per_seed.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)
with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump({"experiment": EXP, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "data_path": DATA_PATH, "n_pooled_complete": int(len(X_all)),
               "n_sites": len(site_keys),
               "column_order_hypothesis": COLNAMES,
               "pearson_triple": [[float(Cfull[i][j]) for j in range(3)] for i in range(3)],
               "strong_pair_rho": RHO_PAIR, "config": _config,
               "code_sha256": CODE_SHA,
               "provenance_scope": "function bytecode + config dict only; not full source text or comments",
               "numpy": np.__version__}, f, indent=2)
print("wrote per_seed.csv, metadata.json")


In [ ]:
# Cell 5 -- Verification from disk; acceptance checks; observations
import csv as _csv
rows = list(_csv.DictReader(open(os.path.join(OUT, "per_seed.csv"))))
assert all(r["experiment"] == "beijing_order_demo" for r in rows), "stamp mismatch (stale file?)"

def vals(cond, target, col):
    return [float(r[col]) for r in rows
            if r["condition"] == cond and r["target"] == target]

REF = single_pair_F(abs(RHO_PAIR))
print(f"{'condition':15s}{'target':18s}{'poly in-sample':>16s}{'poly OOS':>16s}{'RBF (holdout)':>16s}")
for cond in CONDITIONS:
    for t in TARGETS_B:
        p  = vals(cond, t, "frac_poly_D4")
        po = vals(cond, t, "frac_poly_oos")
        rb = vals(cond, t, "frac_rbf_ridge")
        print(f"{cond:15s}{t:18s}{np.mean(p):8.4f}+-{np.std(p):6.4f}"
              f"{np.mean(po):9.4f}+-{np.std(po):6.4f}"
              f"{np.mean(rb):9.4f}+-{np.std(rb):6.4f}")
    print()
print(f"single-pair law at |rho_hat| ({abs(RHO_PAIR):.3f}): {REF:.4f}  [REFERENCE ONLY; law even in rho]")

checks, story = [], []
pc_ok = all(all(x < 0.02 for x in vals(c, "pairwise_control", "frac_rbf_ridge")) for c in CONDITIONS)
checks.append(("pairwise_control: population F2=0 by construction (provable); RBF < 0.02 every condition and seed (empirical class-adequacy threshold)", pc_ok))
m = vals("permuted_gauss", "monomial_c", "frac_poly_oos")
checks.append(("permuted centered monomial -> 1 (provable): OOS poly in (0.85,1.15) each seed",
               all(0.85 < x < 1.15 for x in m)))
m = vals("permuted_gauss", "tanh_prod_c", "frac_poly_oos")
checks.append(("permuted centered tanh product -> 1 (provable): OOS poly in (0.85,1.15) each seed",
               all(0.85 < x < 1.15 for x in m)))
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)

story.append("OBS   permuted in-sample poly (recorded): monomial "
             + " ".join(f"{x:.4f}" for x in vals("permuted_gauss","monomial_c","frac_poly_D4"))
             + "; tanh " + " ".join(f"{x:.4f}" for x in vals("permuted_gauss","tanh_prod_c","frac_poly_D4")))
for cond in ["gauss_scores", "real_std"]:
    story.append(f"OBS   {cond} monomial: in-sample "
                 + " ".join(f"{x:.4f}" for x in vals(cond,"monomial_c","frac_poly_D4"))
                 + f"; OOS {np.mean(vals(cond,'monomial_c','frac_poly_oos')):.4f}"
                 + f"+-{np.std(vals(cond,'monomial_c','frac_poly_oos')):.4f}"
                 + f"  vs single-pair reference {REF:.4f}")
for line in story[len(checks):]:
    print(line)
with open(os.path.join(OUT, "check.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote check.txt")
